# Session 1, Part 1: components, streams, units, and systems

Run each cell with **Shift+Enter**, from the top. Cells marked **Your turn** are exercises. The level says what they need: **1** (change a value), **2** (a few lines of code), **3** (advanced, optional). Every exercise cell already runs as is, so you can continue even if you skip one. Each exercise has a hint, and a solution that stays hidden until you click **Show solution**.

If you know other tools: a `Component` is an entry in the state variable list, a `WasteStream` is a stream with a composition, a `SanUnit` is a unit block, and a `System` is a flowsheet that is solved in order, with recycles converged iteratively.

In [ ]:
# Google Colab only: installs QSDsan the first time. On your own computer this cell does nothing.
# After the install, restart the session (Runtime, Restart session) and run the notebook again from the top.
import sys
if 'google.colab' in sys.modules:
    try:
        import qsdsan
    except ImportError:
        %pip install -q qsdsan==1.5.3 exposan==1.5.3
        print('Installed. Now restart the session (Runtime, Restart session) and run the notebook again from the top.')

In [ ]:
import qsdsan as qs
print(f'QSDsan {qs.__version__}')

# A separate flowsheet keeps the units and streams of this notebook apart from other work.
qs.main_flowsheet.set_flowsheet('s1p1')

## 0. Find your way: reading the documentation

You do not need to remember arguments. Every QSDsan class and function documents itself, and you can read that from where you are working.

| Where you work | Show the documentation | Show the source code |
|---|---|---|
| Jupyter (notebook, JupyterLab) | put `?` after the name, for example `qs.Component?`, or press **Shift+Tab** with the cursor inside the parentheses | `??` after the name |
| Any Python console | `help(qs.Component)` | `import inspect`, then `inspect.getsource(qs.Component)` |
| Spyder | **Ctrl+I** with the cursor on the name (opens the Help pane), or `?` in the console | `??` in the console |
| VS Code | hover over the name | **F12** (Go to Definition) |
| PyCharm | **Ctrl+Q** on the name (**F1** on macOS) | **Ctrl+B** |

Tab completion also helps you explore: type `qs.` or `qs.unit_operations.` and press **Tab**.

In [ ]:
qs.Component?

The documentation of a class starts with what you need at a minimum. For `Component` it says: *"At a minimum, ID, particle_size, degradability, and organic should be provided"*, and the parameter list gives the allowed values (for example `particle_size` must be `'Dissolved gas'`, `'Soluble'`, `'Colloidal'`, or `'Particulate'`). Let us make a component from that:

In [ ]:
X_Cells = qs.Component('X_Cells', formula='C5H7O2N', particle_size='Particulate',
                       degradability='Slowly', organic=True)
print(f'{X_Cells.ID}: MW {X_Cells.MW:.1f} g/mol, COD {X_Cells.i_COD:.2f} g COD/g, N {X_Cells.i_N:.3f} g N/g')

Given a `formula`, QSDsan calculates the molecular weight and the COD and element contents. Without one, an organic component needs you to give its COD (`i_COD`). Error messages are written to tell you what is missing:

In [ ]:
try:
    qs.Component('S_Test', particle_size='Soluble', degradability='Readily', organic=True)
except Exception as e:
    print(f'{type(e).__name__}: {e}')

The same habit works for every key class. This is what you need to create each of them:

| Class | What it represents | What you need to create one | Check |
|---|---|---|---|
| `Component` | one species or state variable (substrate, ammonium, biomass) | `ID`, `particle_size`, `degradability`, `organic`, and a `formula` or `i_COD` | `qs.Component?` |
| `Components` | the set a project uses | a list of components, then `compile()` and `qs.set_thermo(...)` before you make any stream | `qs.Components?` |
| `WasteStream` | the flow of every component | an `ID` and flows by component ID (or an influent model) | `qs.WasteStream?` |
| `SanUnit` | one process unit | `ID`, `ins`, `outs`, and for a new unit type the methods `_run`, `_design`, `_cost` | `qs.SanUnit?` |
| `System` | the units in order, with recycles | `ID`, `path` (the units), and `recycle` if any | `qs.System?` |
| `TEA`, `LCA` | economics and environmental impacts | a simulated system and your assumptions (session 1 part 2) | `qs.TEA?`, `qs.LCA?` |

**Try it (level 1).** Use the documentation to find the allowed values of `degradability`, then make a soluble, readily degradable organic component `S_Sugar` with the formula `C6H12O6`. Print its COD per gram. Then use `?` to look at one other class from the table above.

<details><summary>Hint</summary>

`qs.Component?` lists the allowed values: 'Readily', 'Slowly', or 'Undegradable'.

</details>

In [ ]:
S_Sugar = qs.Component('S_Sugar', formula='C6H12O6', particle_size='Soluble',
                       degradability='Undegradable', organic=True)   # Your turn: fix the degradability
print(f'{S_Sugar.ID}: COD {S_Sugar.i_COD:.2f} g COD/g')

<details><summary>Show solution</summary>

```python
S_Sugar = qs.Component('S_Sugar', formula='C6H12O6', particle_size='Soluble',
                       degradability='Readily', organic=True)
print(f'{S_Sugar.ID}: COD {S_Sugar.i_COD:.2f} g COD/g')
```

</details>

## 1. Components and streams

Every QSDsan project starts by choosing the **components** (the species or state variables), and then telling QSDsan to use them with `set_thermo`. We use the default set.

In [ ]:
cmps = qs.Components.load_default()
qs.set_thermo(cmps)
print(f'{len(cmps)} components, for example: {[c.ID for c in cmps][:8]}')

In [ ]:
# Each component carries properties that QSDsan uses to calculate COD, TKN, solids, etc.
S_F = cmps.S_F
print(S_F.ID, '| COD per gram:', S_F.i_COD, '| particle size:', S_F.particle_size, '| organic:', S_F.organic)

A **`WasteStream`** holds the flow of every component. You can set flows directly, or use a wastewater model that turns a few measured values (COD, TKN, TP) into a full composition.

In [ ]:
ww = qs.WasteStream.codbased_inf_model('ww', flow_tot=1000, units=('L/hr', 'mg/L'), COD=430, TKN=40)
ww.show(N=8)

In [ ]:
# Composite values are calculated from the components.
print(f'COD = {ww.COD:.0f} mg/L, TKN = {ww.TKN:.0f} mg/L, TN = {ww.TN:.0f} mg/L, TP = {ww.TP:.1f} mg/L')
print(f'Total flow = {ww.F_vol*1000:.0f} L/hr, mass flow = {ww.F_mass:.0f} kg/hr')

> **Good to know.** The influent models adjust the N and P content of a few shared components (for example `S_F` and `X_B_Subst`) so that your TKN and TP are matched. Because components are shared, calling a model again with different values changes the TN and TP of streams you made earlier. Create the influent once, then copy it and change flows, as in the exercise below.

**Your turn 1 (level 1).** Make a stronger wastewater: copy `ww` as `ww_strong` and double the flow of the soluble organics `S_F`. Print the COD of both streams and the change. Then print the concentration of `S_F` in each.

<details><summary>Hint</summary>

`imass` is indexed by component ID and is in kg/hr. `iconc` gives the concentration in mg/L. `copy` makes an independent stream. To change one component's flow, for example: `ww_strong.imass['S_F'] *= 3` triples it.

</details>

In [ ]:
ww_strong = ww.copy('ww_strong')
# Your turn: double the flow of S_F in ww_strong.
print(f'COD: ww {ww.COD:.0f} mg/L, ww_strong {ww_strong.COD:.0f} mg/L')

<details><summary>Show solution</summary>

```python
ww_strong = ww.copy('ww_strong')
ww_strong.imass['S_F'] *= 2
print(f'COD: ww {ww.COD:.0f} mg/L, ww_strong {ww_strong.COD:.0f} mg/L')
print(f"S_F: ww {ww.iconc['S_F']:.0f} mg/L, ww_strong {ww_strong.iconc['S_F']:.0f} mg/L")
```

</details>

## 2. Units and systems

A **`SanUnit`** turns input streams into output streams (`_run`), and can also compute its size (`_design`) and cost (`_cost`). QSDsan ships many unit operations in `qs.unit_operations`. Here we build a small activated sludge layout from existing units: an anoxic tank, an aerated tank, and a clarifier that sends sludge back to the first tank.

Streams and units connect with **pipe notation**: `A1-0` is the first outlet of `A1`, and `1-A1` is the second inlet of `A1`.

In [ ]:
# Air for the aerated tank (illustrative numbers)
air = qs.WasteStream('air', S_O2=ww.F_mass*0.01)
air.imol['S_N2'] = air.imol['S_O2']/0.21*0.79

A1 = qs.unit_operations.MixTank('A1', ins=(ww, 'recycle'), tau=2, V_wf=0.8, init_with='WasteStream')
O1 = qs.unit_operations.MixTank('O1', ins=(A1-0, air), tau=4, V_wf=0.8, init_with='WasteStream')

# The clarifier is modeled as a splitter: 90% of dissolved material goes with the liquid (effluent),
# and all particulates go to the sludge. A second splitter then wastes 10% of the sludge (WAS)
# and returns the rest to the first tank (the recycle).
split = [0.9 if c.particle_size != 'Particulate' else 0 for c in cmps]   # one number per component
C1 = qs.unit_operations.Splitter('C1', ins=O1-0, outs=('effluent', 'sludge'), split=split, init_with='WasteStream')
S1 = qs.unit_operations.Splitter('S1', ins=C1-1, outs=('WAS', 1-A1), split=0.1, init_with='WasteStream')

In [ ]:
# A system lists the units in order and names the recycle stream.
sys = qs.System('sys', path=(A1, O1, C1, S1), recycle=S1-1)
# Solids build up slowly in a recycle, so the default tolerance stops too early.
# We converge it tightly so that small changes are visible (see tutorial 6 on convergence).
sys.set_tolerance(rmol=1e-6, mol=1e-6, maxiter=200)
sys.simulate()
sys.diagram()

Results come from the units and the system. `sys.results()` summarizes each unit's design and cost, and each stream can be inspected directly.

In [ ]:
sys.results()

In [ ]:
eff = qs.F.stream.effluent   # units and streams are registered by ID in the flowsheet (qs.F)
was = qs.F.stream.WAS
print(f'Effluent: {eff.F_vol*1000:.0f} L/hr, COD {eff.COD:.0f} mg/L, TSS {eff.get_TSS():.0f} mg/L')
print(f'Waste sludge (WAS): {was.F_vol*1000:.0f} L/hr, TSS {was.get_TSS():.0f} mg/L')
print(f'Recycle: {S1.outs[1].F_vol*1000:.0f} L/hr')
print(f'Purchase cost of the equipment: {sys.purchase_cost:,.0f} USD')

**Your turn 2 (level 1).** The clarifier above sends 90% of the dissolved material (including water) to the effluent. Change that to 50%, re-simulate, and compare the effluent flow, the recycle flow, and the waste sludge (WAS) flow with the values above. The split gives the fraction of each component that goes to the *first* outlet (the effluent).

<details><summary>Hint</summary>

`C1.split` is a list with one number per component, in the order of `cmps`. The effluent COD concentration does not change, because all dissolved components are split by the same fraction; the flows do.

</details>

In [ ]:
# The split is a list in the order of the components: the fraction sent to the first outlet.
split_new = [0.9 if c.particle_size != 'Particulate' else 0 for c in cmps]   # Your turn: change 0.9
C1.split = split_new
sys.simulate()
print(f'Effluent {eff.F_vol*1000:.0f} L/hr, WAS {was.F_vol*1000:.0f} L/hr, recycle {S1.outs[1].F_vol*1000:.0f} L/hr')

<details><summary>Show solution</summary>

```python
split_new = [0.5 if c.particle_size != 'Particulate' else 0 for c in cmps]
C1.split = split_new
sys.simulate()
print(f'Effluent {eff.F_vol*1000:.0f} L/hr, WAS {was.F_vol*1000:.0f} L/hr, recycle {S1.outs[1].F_vol*1000:.0f} L/hr')
# Restore the original split so that later cells match the text.
C1.split = split
sys.simulate()
```

</details>

**Your turn 3 (level 2).** Add a second aerated tank `O2` after `O1` (with its own air stream), and build a new system `sys2` with the units in the right order and the same recycle. Compare the purchase cost with `sys`.

<details><summary>Hint</summary>

A system needs its own copies of the streams and its own units, so make copies of the influent and air.

</details>

In [ ]:
# Your turn: create air2 and O2, then build sys2 = qs.System('sys2', path=(...), recycle=...)
# Tip: if you reuse a unit in a new system, give the new units new IDs, and mind the connections.
print('Baseline purchase cost:', round(sys.purchase_cost))

<details><summary>Show solution</summary>

```python
air2 = air.copy('air2')
A1b = qs.unit_operations.MixTank('A1b', ins=(ww.copy('ww_b'), 'recycle_b'), tau=2, V_wf=0.8, init_with='WasteStream')
O1b = qs.unit_operations.MixTank('O1b', ins=(A1b-0, air.copy('air_b')), tau=4, V_wf=0.8, init_with='WasteStream')
O2b = qs.unit_operations.MixTank('O2b', ins=(O1b-0, air2), tau=4, V_wf=0.8, init_with='WasteStream')
C1b = qs.unit_operations.Splitter('C1b', ins=O2b-0, outs=('effluent_b', 'sludge_b'), split=split, init_with='WasteStream')
S1b = qs.unit_operations.Splitter('S1b', ins=C1b-1, outs=('WAS_b', 1-A1b), split=0.1, init_with='WasteStream')
sys2 = qs.System('sys2', path=(A1b, O1b, O2b, C1b, S1b), recycle=S1b-1)
sys2.set_tolerance(rmol=1e-6, mol=1e-6, maxiter=200)
sys2.simulate()
print(f'Purchase cost: baseline {sys.purchase_cost:,.0f} USD, with second aerated tank {sys2.purchase_cost:,.0f} USD')
```

</details>

## 3. Write your own unit

What if QSDsan doesn't have the unit operations you need? A custom unit is a Python class with a few methods: `_run` (mass balance), `_design` (sizing), and `_cost` (equipment cost). Below is a very simple aerobic reactor that converts a fraction of methanol to CO2.

We first add O2 and CO2 to the component set, because this reaction needs them. Because the thermo changes, the streams above keep working but new streams use the new set.

In [ ]:
kwargs = dict(particle_size='Dissolved gas', degradability='Undegradable', organic=False)
O2 = qs.Component('O2', search_ID='O2', **kwargs)
CO2 = qs.Component('CO2', search_ID='CO2', **kwargs)
cmps2 = qs.Components([*qs.Components.load_default(), O2, CO2])
cmps2.compile(ignore_inaccurate_molar_weight=True)
qs.set_thermo(cmps2)

In [ ]:
class AerobicReactor(qs.SanUnit):
    '''Converts a fraction of methanol (S_CH3OH) to CO2 with oxygen from aeration.'''
    _N_ins = 2    # wastewater and air (O2)
    _N_outs = 2   # treated water and CO2
    _units = {'Volume': 'm3', 'Stainless steel': 'kg'}

    def __init__(self, ID='', ins=None, outs=(), thermo=None, init_with='WasteStream',
                 conversion=0.9, aeration_rate=5, HRT=5):
        qs.SanUnit.__init__(self, ID, ins, outs, thermo, init_with)
        self.conversion = conversion        # fraction of methanol converted
        self.aeration_rate = aeration_rate  # kg O2 per m3 of wastewater
        self.HRT = HRT                      # hydraulic retention time, hr
        self.F_BM = {'Tank': 2}             # bare module factor

    def _run(self):
        inf, o2 = self.ins
        eff, co2 = self.outs
        o2.phase = co2.phase = 'g'
        o2.imass['O2'] = self.aeration_rate * self.F_vol_in
        eff.mix_from(self.ins)
        eff.imass['S_O2'] = eff.imass['O2']   # dissolved oxygen
        eff.imass['O2'] = 0
        # CH3OH + 1.5 O2 -> CO2 + 2 H2O
        converted = self.conversion * inf.imol['S_CH3OH']
        eff.imol['S_CH3OH'] -= converted
        eff.imol['S_O2'] -= 1.5 * converted
        eff.imol['H2O'] += 2 * converted
        co2.imol['CO2'] = converted

    def _design(self):
        D = self.design_results
        D['Volume'] = self.outs[0].F_vol * self.HRT / 0.8          # 80% working volume
        D['Stainless steel'] = 100 * D['Volume']                   # kg, a rough rule of thumb

    def _cost(self):
        self.baseline_purchase_costs['Tank'] = 3 * self.design_results['Stainless steel']   # 3 USD/kg

In [ ]:
methanol = qs.WasteStream('methanol', H2O=1000, S_CH3OH=0.5, units='kg/hr')
R1 = AerobicReactor('R1', ins=(methanol, 'o2'), outs=('treated', 'co2'))
R1.simulate()
R1.show()

In [ ]:
R1.results()

**Your turn 4 (level 2).** Change the conversion to 0.5 and re-simulate. How do the methanol in the effluent and the CO2 produced change? Then print the tank volume and the purchase cost.

<details><summary>Hint</summary>

Results for `design_results`, `purchase_cost`, and the streams are updated by `simulate()`.

</details>

In [ ]:
R1.conversion = 0.9   # Your turn: change this value
R1.simulate()
print(f"Effluent methanol: {R1.outs[0].imass['S_CH3OH']:.3f} kg/hr, CO2: {R1.outs[1].imass['CO2']:.3f} kg/hr")

<details><summary>Show solution</summary>

```python
R1.conversion = 0.5
R1.simulate()
print(f"Effluent methanol: {R1.outs[0].imass['S_CH3OH']:.3f} kg/hr, CO2: {R1.outs[1].imass['CO2']:.3f} kg/hr")
print(f"Tank volume: {R1.design_results['Volume']:.1f} m3, purchase cost: {R1.purchase_cost:,.0f} USD")
```

</details>

**Your turn 5 (level 3).** Extend the reactor: add electricity use for aeration proportional to the airflow (assume 0.02 kW per kg/hr of O2), by setting `self.power_utility.consumption` in `_cost`. Create the subclass `AerobicReactorP`, simulate it, and read the power from the results.

<details><summary>Hint</summary>

Use `super()._cost()` to keep the tank cost, then add the power line.

</details>

In [ ]:
# Your turn: subclass AerobicReactor and override _cost.
class AerobicReactorP(AerobicReactor):
    pass

R2 = AerobicReactorP('R2', ins=(methanol.copy('methanol2'), 'o2_2'), outs=('treated2', 'co2_2'))
R2.simulate()
R2.results()

<details><summary>Show solution</summary>

```python
class AerobicReactorP(AerobicReactor):
    def _cost(self):
        super()._cost()
        self.power_utility.consumption = 0.02 * self.ins[1].F_mass   # kW

R2 = AerobicReactorP('R2', ins=(methanol.copy('methanol2'), 'o2_2'), outs=('treated2', 'co2_2'))
R2.simulate()
print(f'Power: {R2.power_utility.consumption:.2f} kW')
R2.results()
```

</details>

## 4. Look at a ready-made project

EXPOsan collects systems built with QSDsan. Each one is a Python package with a similar pattern, which is also a good template for your own project. We look at `bsm1`, the Benchmark Simulation Model No. 1: a standard activated sludge plant used to compare process models and control strategies. We will simulate it dynamically in session 2.

In [ ]:
import os
from exposan import bsm1
folder = os.path.dirname(bsm1.__file__)
print('Folder:', folder)
print([f for f in sorted(os.listdir(folder)) if not f.startswith('__')])

Typical contents: `system.py` builds the components, units and the system, `model.py` and `analyses.py` hold analyses built on the system (for example uncertainty analysis, session 2), `data/` has inputs and parameters, and `README.rst` describes the study. A `load()` function builds everything on demand. The same pattern works for your own project: the system in one module, the analyses in another, and your data in `data/`.

The first `load()` can take about 20 seconds because some of the code is compiled.

In [ ]:
bsm1.load()
bsm1.sys.diagram()

In [ ]:
# The units in the system and their classes. This system is dynamic: we simulate it in session 2.
[(u.ID, type(u).__name__) for u in bsm1.sys.units]

**Your turn 6 (level 1).** Load another EXPOsan system, `cas` (a conventional activated sludge plant), and show its diagram. Compare its layout with `bsm1`. The cell below also lists the EXPOsan modules available.

<details><summary>Hint</summary>

Import the module with `from exposan import cas`, call `cas.load()`, and the system is `cas.sys`. A warning about `sludge_moisture` appears when loading `cas`; it is harmless.

</details>

In [ ]:
import pkgutil, exposan
modules = [m.name for m in pkgutil.iter_modules(exposan.__path__) if m.ispkg]
print(modules)
# Your turn: load cas and show its diagram. The pattern is the same as for bsm1.

<details><summary>Show solution</summary>

```python
import pkgutil, exposan
modules = [m.name for m in pkgutil.iter_modules(exposan.__path__) if m.ispkg]
print(modules)
from exposan import cas
cas.load()
cas.sys.diagram()
```

</details>

## Where to go next

- Tutorials: [Components](https://qsdsan.readthedocs.io/en/latest/tutorials/2_Component.html), [WasteStream](https://qsdsan.readthedocs.io/en/latest/tutorials/3_WasteStream.html), [SanUnit (basic)](https://qsdsan.readthedocs.io/en/latest/tutorials/4_SanUnit_basic.html), [SanUnit (advanced)](https://qsdsan.readthedocs.io/en/latest/tutorials/5_SanUnit_advanced.html), [System](https://qsdsan.readthedocs.io/en/latest/tutorials/6_System.html), [Modeling notes and pitfalls](https://qsdsan.readthedocs.io/en/latest/tutorials/14_Modeling_Notes_and_Pitfalls.html).
- Next in this session: cost and environmental impacts (TEA and LCA).